In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import re
import pickle

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [3]:
players_raw = pd.read_csv("../data/scrape/amateur/players.csv")
matches = pd.read_csv("../data/scrape/amateur/matches.csv")
player_stats = pd.read_csv("../data/scrape/amateur/player_stats.csv")
players = pd.read_csv("../data/scrape/amateur/players.csv")

In [4]:
def standardize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

matches = standardize_columns(matches)
player_stats = standardize_columns(player_stats)
players = standardize_columns(players)

print("matches columns:", matches.columns.tolist())
print("player_stats columns:", player_stats.columns.tolist())
print("players columns:", players.columns.tolist())

matches columns: ['match_id', 'season', 'game_date', 'league', 'home_club_id', 'away_club_id', 'home_goals', 'away_goals']
player_stats columns: ['player_id', 'match_id', 'club_id', 'goals', 'assists', 'yellow', 'yellow_red', 'red', 'start_eleven', 'minutes', 'on_min', 'off_min', 'team_goals', 'team_conceded', 'rating']
players columns: ['player_id', 'player_name', 'nationality', 'date_of_birth', 'height', 'position']


In [5]:
def pick_col(df, candidates, required=True):
    cols = list(df.columns)

    for c in candidates:
        if c in cols:
            return c

    for c in candidates:
        for col in cols:
            if c in col:
                return col

    if required:
        raise KeyError(f"Keine passende Spalte gefunden. Kandidaten: {candidates}")
    return None


PS = {
    "player_id": pick_col(player_stats, ["player_id"]),
    "match_id": pick_col(player_stats, ["match_id"]),
    "club_id": pick_col(player_stats, ["club_id"]),
    "season": pick_col(player_stats, ["season"], required=False),
    "goals": pick_col(player_stats, ["goals"], required=False),
    "assists": pick_col(player_stats, ["assists"], required=False),
    "yellow_cards": pick_col(player_stats, ["yellow"], required=False),
    "red_cards": pick_col(player_stats, ["red"], required=False),
    "yellow_red_cards": pick_col(player_stats, ["yellow_red"], required=False),
    "started": pick_col(player_stats, ["start_eleven"], required=False),
    "minutes": pick_col(player_stats, ["minutes"], required=False),
    "rating": pick_col(player_stats, ["rating"], required=False),
    "team_goals": pick_col(player_stats, ["team_goals"], required=False),
    "team_conceded": pick_col(player_stats, ["team_conceded"], required=False),
}

M = {
    "match_id": pick_col(matches, ["match_id"]),
    "season": pick_col(matches, ["season"], required=False),
    "competition": pick_col(matches, ["league"], required=False),
    "home_club_id": pick_col(matches, ["home_club_id"]),
    "away_club_id": pick_col(matches, ["away_club_id"]),
    "home_goals": pick_col(matches, ["home_goals"]),
    "away_goals": pick_col(matches, ["away_goals"]),
    "match_date": pick_col(matches, ["game_date"], required=False),
}

P = {
    "player_id": pick_col(players, ["player_id"]),
    "birth_date": pick_col(players, ["date_of_birth"]),
    "position": pick_col(players, ["position"], required=False),
}

print("PS:", PS)
print("M:", M)
print("P:", P)

PS: {'player_id': 'player_id', 'match_id': 'match_id', 'club_id': 'club_id', 'season': None, 'goals': 'goals', 'assists': 'assists', 'yellow_cards': 'yellow', 'red_cards': 'red', 'yellow_red_cards': 'yellow_red', 'started': 'start_eleven', 'minutes': 'minutes', 'rating': 'rating', 'team_goals': 'team_goals', 'team_conceded': 'team_conceded'}
M: {'match_id': 'match_id', 'season': 'season', 'competition': 'league', 'home_club_id': 'home_club_id', 'away_club_id': 'away_club_id', 'home_goals': 'home_goals', 'away_goals': 'away_goals', 'match_date': 'game_date'}
P: {'player_id': 'player_id', 'birth_date': 'date_of_birth', 'position': 'position'}


In [6]:
player_stats = player_stats.rename(columns={v: k for k, v in PS.items() if v is not None})
matches = matches.rename(columns={v: k for k, v in M.items() if v is not None})
players = players.rename(columns={v: k for k, v in P.items() if v is not None})

print(player_stats.columns.tolist())
print(matches.columns.tolist())
print(players.columns.tolist())

['player_id', 'match_id', 'club_id', 'goals', 'assists', 'yellow_cards', 'yellow_red_cards', 'red_cards', 'started', 'minutes', 'on_min', 'off_min', 'team_goals', 'team_conceded', 'rating']
['match_id', 'season', 'match_date', 'competition', 'home_club_id', 'away_club_id', 'home_goals', 'away_goals']
['player_id', 'player_name', 'nationality', 'birth_date', 'height', 'position']


In [7]:
def season_to_start_year(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    m = re.fullmatch(r"(\d{2})\s*[/\-]\s*(\d{2})", s)
    if m:
        return 2000 + int(m.group(1))

    m = re.fullmatch(r"(20\d{2})\s*[/\-]\s*(\d{2})", s)
    if m:
        return int(m.group(1))

    m = re.fullmatch(r"(20\d{2})\s*[/\-]\s*(20\d{2})", s)
    if m:
        return int(m.group(1))

    m = re.fullmatch(r"20\d{2}", s)
    if m:
        return int(s)

    return np.nan


def age_on_aug1(season_start, birth_date):
    if pd.isna(season_start) or pd.isna(birth_date):
        return np.nan

    cutoff = pd.Timestamp(year=int(season_start), month=8, day=1)

    return cutoff.year - birth_date.year - (
        (cutoff.month, cutoff.day) < (birth_date.month, birth_date.day)
    )


def to_binary_started(x):
    if pd.isna(x):
        return 0

    if isinstance(x, (bool, np.bool_)):
        return int(x)

    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(x > 0)

    s = str(x).strip().lower()
    if s in {"1", "true", "yes", "y"}:
        return 1
    return 0


def normalize_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def is_first_liga(comp):
    s = normalize_text(comp)
    return int(bool(re.search(r"\b1\.?\s*liga\b", s)))


def is_pl(comp):
    s = normalize_text(comp)
    return int(("promotion league" in s) or bool(re.search(r"\bpl\b", s)))


def get_result(row):
    if row["club_id"] == row["home_club_id"]:
        if row["home_goals"] > row["away_goals"]:
            return "win"
        elif row["home_goals"] < row["away_goals"]:
            return "loss"
        else:
            return "draw"

    elif row["club_id"] == row["away_club_id"]:
        if row["away_goals"] > row["home_goals"]:
            return "win"
        elif row["away_goals"] < row["home_goals"]:
            return "loss"
        else:
            return "draw"

    else:
        return None

In [8]:
for col in [
    "goals",
    "assists",
    "yellow_cards",
    "red_cards",
    "yellow_red_cards",
    "minutes",
    "rating",
    "team_goals",
    "team_conceded"
]:
    if col not in player_stats.columns:
        player_stats[col] = 0

for col in ["home_goals", "away_goals"]:
    if col not in matches.columns:
        matches[col] = np.nan

player_stats["rating"] = (
    player_stats["rating"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

num_cols_player_stats = [
    "goals",
    "assists",
    "yellow_cards",
    "red_cards",
    "yellow_red_cards",
    "minutes",
    "rating",
    "team_goals",
    "team_conceded"
]

for col in num_cols_player_stats:
    player_stats[col] = pd.to_numeric(player_stats[col], errors="coerce")

matches["home_goals"] = pd.to_numeric(matches["home_goals"], errors="coerce")
matches["away_goals"] = pd.to_numeric(matches["away_goals"], errors="coerce")

player_stats["started"] = player_stats["started"].apply(to_binary_started)
players["birth_date"] = pd.to_datetime(players["birth_date"], errors="coerce")

In [9]:
match_cols = ["match_id", "home_club_id", "away_club_id", "home_goals", "away_goals"]

if "season" in matches.columns:
    match_cols.append("season")
if "competition" in matches.columns:
    match_cols.append("competition")
if "match_date" in matches.columns:
    match_cols.append("match_date")

match_join = matches[match_cols].copy()

if "season" in match_join.columns:
    match_join = match_join.rename(columns={"season": "season_match"})

player_join = players[["player_id", "birth_date", "position"]].drop_duplicates(subset=["player_id"]).copy()

df = player_stats.merge(match_join, on="match_id", how="left")
df = df.merge(player_join, on="player_id", how="left")

if "season" not in df.columns:
    df["season"] = df["season_match"]
else:
    df["season"] = df["season"].fillna(df.get("season_match"))

df["season"] = df["season"].apply(season_to_start_year)
df = df.drop_duplicates(subset=["player_id", "match_id", "club_id"]).copy()

print("df shape:", df.shape)
display(df.head())

df shape: (146964, 25)


,player_id,match_id,club_id,goals,assists,yellow_cards,yellow_red_cards,red_cards,started,minutes,on_min,off_min,team_goals,team_conceded,rating,home_club_id,away_club_id,home_goals,away_goals,season_match,competition,match_date,birth_date,position,season
0,284695,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.5,322,8508,2,0,20/21,pl,2020-08-15,1995-06-13,Torwart,2020
1,115188,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.4,322,8508,2,0,20/21,pl,2020-08-15,1991-01-11,Innenverteidiger,2020
2,19279,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.4,322,8508,2,0,20/21,pl,2020-08-15,1986-01-03,Innenverteidiger,2020
3,126514,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.3,322,8508,2,0,20/21,pl,2020-08-15,1993-03-03,Linker Verteidiger,2020
4,267582,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.1,322,8508,2,0,20/21,pl,2020-08-15,1992-04-06,Rechter Verteidiger,2020


In [10]:
df_2526 = df[df["season"] == 2025].copy()

print("Rows 25/26:", df_2526.shape[0])
print("Players 25/26:", df_2526["player_id"].nunique())
display(df_2526.head())

Rows 25/26: 19829
Players 25/26: 1848


,player_id,match_id,club_id,goals,assists,yellow_cards,yellow_red_cards,red_cards,started,minutes,on_min,off_min,team_goals,team_conceded,rating,home_club_id,away_club_id,home_goals,away_goals,season_match,competition,match_date,birth_date,position,season
38314,816725,4645719,15446,0,0,NaN,NaN,NaN,0,90,0,0,5,1,6.7,15446,49621,5,1,25/26,pl,2025-08-02,2005-03-24,Torwart,2025
38315,902783,4645719,15446,0,1,NaN,NaN,NaN,0,90,0,0,5,1,7.2,15446,49621,5,1,25/26,pl,2025-08-02,2005-05-04,Innenverteidiger,2025
38316,607588,4645719,15446,0,0,NaN,NaN,NaN,0,90,0,0,5,1,7.0,15446,49621,5,1,25/26,pl,2025-08-02,2002-09-24,Innenverteidiger,2025
38317,1060759,4645719,15446,1,0,NaN,NaN,NaN,0,90,0,0,5,1,7.7,15446,49621,5,1,25/26,pl,2025-08-02,2004-05-13,Innenverteidiger,2025
38318,864381,4645719,15446,0,0,NaN,NaN,NaN,0,90,0,0,5,1,6.8,15446,49621,5,1,25/26,pl,2025-08-02,2005-02-21,Rechter Verteidiger,2025


In [11]:
df_2526["alter"] = df_2526.apply(lambda row: age_on_aug1(row["season"], row["birth_date"]), axis=1)

df_2526["result"] = df_2526.apply(get_result, axis=1)

df_2526["is_win"] = (df_2526["result"] == "win").astype(int)
df_2526["is_draw"] = (df_2526["result"] == "draw").astype(int)
df_2526["is_loss"] = (df_2526["result"] == "loss").astype(int)

df_2526["is_first_liga"] = df_2526["competition"].apply(is_first_liga)
df_2526["is_pl"] = df_2526["competition"].apply(is_pl)

display(df_2526.head())

,player_id,match_id,club_id,goals,assists,yellow_cards,yellow_red_cards,red_cards,started,minutes,on_min,off_min,team_goals,team_conceded,rating,home_club_id,away_club_id,home_goals,away_goals,season_match,competition,match_date,birth_date,position,season,alter,result,is_win,is_draw,is_loss,is_first_liga,is_pl
38314,816725,4645719,15446,0,0,NaN,NaN,NaN,0,90,0,0,5,1,6.7,15446,49621,5,1,25/26,pl,2025-08-02,2005-03-24,Torwart,2025,20.0,win,1,0,0,0,1
38315,902783,4645719,15446,0,1,NaN,NaN,NaN,0,90,0,0,5,1,7.2,15446,49621,5,1,25/26,pl,2025-08-02,2005-05-04,Innenverteidiger,2025,20.0,win,1,0,0,0,1
38316,607588,4645719,15446,0,0,NaN,NaN,NaN,0,90,0,0,5,1,7.0,15446,49621,5,1,25/26,pl,2025-08-02,2002-09-24,Innenverteidiger,2025,22.0,win,1,0,0,0,1
38317,1060759,4645719,15446,1,0,NaN,NaN,NaN,0,90,0,0,5,1,7.7,15446,49621,5,1,25/26,pl,2025-08-02,2004-05-13,Innenverteidiger,2025,21.0,win,1,0,0,0,1
38318,864381,4645719,15446,0,0,NaN,NaN,NaN,0,90,0,0,5,1,6.8,15446,49621,5,1,25/26,pl,2025-08-02,2005-02-21,Rechter Verteidiger,2025,20.0,win,1,0,0,0,1


In [12]:
club_season_matches = (
    df_2526[["season", "club_id", "match_id"]]
    .drop_duplicates()
    .groupby(["season", "club_id"], as_index=False)
    .agg(team_spiele_saison=("match_id", "nunique"))
)

player_season_clubs = (
    df_2526[["player_id", "season", "club_id"]]
    .drop_duplicates()
    .merge(club_season_matches, on=["season", "club_id"], how="left")
)

player_season_team_matches = (
    player_season_clubs
    .groupby(["player_id", "season"], as_index=False)
    .agg(team_spiele_saison=("team_spiele_saison", "sum"))
)

display(player_season_team_matches.head())

,player_id,season,team_spiele_saison
0,33056,2025,19
1,37522,2025,19
2,44857,2025,22
3,59232,2025,19
4,59991,2025,19


In [13]:
season_features_2526 = (
    df_2526.groupby(["player_id", "season"], as_index=False)
      .agg(
          alter=("alter", "first"),
          position=("position", "first"),

          anzahl_spiele=("match_id", "nunique"),

          tore_abs=("goals", "sum"),
          assists_abs=("assists", "sum"),

          rote_karten_abs=("red_cards", "sum"),
          gelbe_karten_abs=("yellow_cards", "sum"),
          gelb_rote_karten_abs=("yellow_red_cards", "sum"),

          startelf_abs=("started", "sum"),
          minuten_abs=("minutes", "sum"),

          team_tore_abs=("team_goals", "sum"),
          team_gegentore_abs=("team_conceded", "sum"),

          siege_abs=("is_win", "sum"),
          unentschieden_abs=("is_draw", "sum"),
          niederlagen_abs=("is_loss", "sum"),

          prozent_1_liga=("is_first_liga", "mean"),
          prozent_pl=("is_pl", "mean"),

          avg_rating_current=("rating", "mean")
      )
)

season_features_2526 = season_features_2526.merge(
    player_season_team_matches,
    on=["player_id", "season"],
    how="left"
)

season_features_2526["einsatzquote"] = (
    season_features_2526["anzahl_spiele"] / season_features_2526["team_spiele_saison"]
).clip(lower=0, upper=1)

season_features_2526["prozent_1_liga"] = season_features_2526["prozent_1_liga"] * 100
season_features_2526["prozent_pl"] = season_features_2526["prozent_pl"] * 100

season_features_2526["tore_pro_spiel"] = season_features_2526["tore_abs"] / season_features_2526["anzahl_spiele"]
season_features_2526["assists_pro_spiel"] = season_features_2526["assists_abs"] / season_features_2526["anzahl_spiele"]

season_features_2526["rote_karten_pro_spiel"] = season_features_2526["rote_karten_abs"] / season_features_2526["anzahl_spiele"]
season_features_2526["gelbe_karten_pro_spiel"] = season_features_2526["gelbe_karten_abs"] / season_features_2526["anzahl_spiele"]
season_features_2526["gelb_rote_karten_pro_spiel"] = season_features_2526["gelb_rote_karten_abs"] / season_features_2526["anzahl_spiele"]

season_features_2526["startelf_pro_spiel"] = season_features_2526["startelf_abs"] / season_features_2526["anzahl_spiele"]
season_features_2526["minuten_pro_spiel"] = season_features_2526["minuten_abs"] / season_features_2526["anzahl_spiele"]

season_features_2526["team_tore_pro_spiel"] = season_features_2526["team_tore_abs"] / season_features_2526["anzahl_spiele"]
season_features_2526["team_gegentore_pro_spiel"] = season_features_2526["team_gegentore_abs"] / season_features_2526["anzahl_spiele"]

season_features_2526["siege_pro_spiel"] = season_features_2526["siege_abs"] / season_features_2526["anzahl_spiele"]
season_features_2526["unentschieden_pro_spiel"] = season_features_2526["unentschieden_abs"] / season_features_2526["anzahl_spiele"]
season_features_2526["niederlagen_pro_spiel"] = season_features_2526["niederlagen_abs"] / season_features_2526["anzahl_spiele"]

print(season_features_2526.shape)
display(season_features_2526.head())

(1848, 34)


,player_id,season,alter,position,anzahl_spiele,tore_abs,assists_abs,rote_karten_abs,gelbe_karten_abs,gelb_rote_karten_abs,startelf_abs,minuten_abs,team_tore_abs,team_gegentore_abs,siege_abs,unentschieden_abs,niederlagen_abs,prozent_1_liga,prozent_pl,avg_rating_current,team_spiele_saison,einsatzquote,tore_pro_spiel,assists_pro_spiel,rote_karten_pro_spiel,gelbe_karten_pro_spiel,gelb_rote_karten_pro_spiel,startelf_pro_spiel,minuten_pro_spiel,team_tore_pro_spiel,team_gegentore_pro_spiel,siege_pro_spiel,unentschieden_pro_spiel,niederlagen_pro_spiel
0,33056,2025,38.0,Zentrales Mittelfeld,13,0,0,0.0,0.0,0.0,0,833,12,15,5,3,5,0.0,0.0,6.761538,19,0.684211,0.000000,0.000000,0.0,0.0,0.0,0.0,64.076923,0.923077,1.153846,0.384615,0.230769,0.384615
1,37522,2025,37.0,Defensives Mittelfeld,17,0,1,0.0,0.0,0.0,0,1492,18,36,5,3,9,0.0,0.0,6.982353,19,0.894737,0.000000,0.058824,0.0,0.0,0.0,0.0,87.764706,1.058824,2.117647,0.294118,0.176471,0.529412
2,44857,2025,34.0,Rechtsaußen,19,4,3,0.0,0.0,0.0,0,1426,31,18,12,3,4,0.0,100.0,7.057895,22,0.863636,0.210526,0.157895,0.0,0.0,0.0,0.0,75.052632,1.631579,0.947368,0.631579,0.157895,0.210526
3,59232,2025,38.0,Linksaußen,16,2,1,0.0,0.0,0.0,0,1104,18,18,4,8,4,0.0,0.0,6.875000,19,0.842105,0.125000,0.062500,0.0,0.0,0.0,0.0,69.000000,1.125000,1.125000,0.250000,0.500000,0.250000
4,59991,2025,37.0,Rechtes Mittelfeld,18,15,0,0.0,0.0,0.0,0,1289,28,37,7,3,8,0.0,0.0,7.627778,19,0.947368,0.833333,0.000000,0.0,0.0,0.0,0.0,71.611111,1.555556,2.055556,0.388889,0.166667,0.444444


In [14]:
predict_columns = [
    "player_id",
    "season",
    "alter",
    "position",

    "avg_rating_current",
    "einsatzquote",

    "tore_pro_spiel",
    "tore_abs",

    "assists_pro_spiel",
    "assists_abs",

    "rote_karten_pro_spiel",
    "gelbe_karten_pro_spiel",
    "gelb_rote_karten_pro_spiel",

    "rote_karten_abs",
    "gelbe_karten_abs",
    "gelb_rote_karten_abs",

    "startelf_pro_spiel",
    "startelf_abs",

    "minuten_pro_spiel",
    "minuten_abs",

    "anzahl_spiele",

    "team_tore_pro_spiel",
    "team_tore_abs",

    "team_gegentore_pro_spiel",
    "team_gegentore_abs",

    "siege_pro_spiel",
    "unentschieden_pro_spiel",
    "niederlagen_pro_spiel",

    "siege_abs",
    "unentschieden_abs",
    "niederlagen_abs",

    "prozent_1_liga",
    "prozent_pl",
]

predict_dataset = season_features_2526[predict_columns].copy()

predict_dataset["position"] = predict_dataset["position"].fillna("unknown").astype(str).str.strip()

predict_dataset = pd.get_dummies(
    predict_dataset,
    columns=["position"],
    prefix="position",
    dtype=int
)

print(predict_dataset.shape)
display(predict_dataset.head())

(1848, 49)


,player_id,season,alter,avg_rating_current,einsatzquote,tore_pro_spiel,tore_abs,assists_pro_spiel,assists_abs,rote_karten_pro_spiel,gelbe_karten_pro_spiel,gelb_rote_karten_pro_spiel,rote_karten_abs,gelbe_karten_abs,gelb_rote_karten_abs,startelf_pro_spiel,startelf_abs,minuten_pro_spiel,minuten_abs,anzahl_spiele,team_tore_pro_spiel,team_tore_abs,team_gegentore_pro_spiel,team_gegentore_abs,siege_pro_spiel,unentschieden_pro_spiel,niederlagen_pro_spiel,siege_abs,unentschieden_abs,niederlagen_abs,prozent_1_liga,prozent_pl,position_Abwehr,position_Defensives Mittelfeld,position_Hängende Spitze,position_Innenverteidiger,position_Linker Verteidiger,position_Linkes Mittelfeld,position_Linksaußen,position_Mittelfeld,position_Mittelstürmer,position_Offensives Mittelfeld,position_Rechter Verteidiger,position_Rechtes Mittelfeld,position_Rechtsaußen,position_Sturm,position_Torwart,position_Zentrales Mittelfeld,position_unknown
0,33056,2025,38.0,6.761538,0.684211,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,64.076923,833,13,0.923077,12,1.153846,15,0.384615,0.230769,0.384615,5,3,5,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,37522,2025,37.0,6.982353,0.894737,0.000000,0,0.058824,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,87.764706,1492,17,1.058824,18,2.117647,36,0.294118,0.176471,0.529412,5,3,9,0.0,0.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,44857,2025,34.0,7.057895,0.863636,0.210526,4,0.157895,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,75.052632,1426,19,1.631579,31,0.947368,18,0.631579,0.157895,0.210526,12,3,4,0.0,100.0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,59232,2025,38.0,6.875000,0.842105,0.125000,2,0.062500,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,69.000000,1104,16,1.125000,18,1.125000,18,0.250000,0.500000,0.250000,4,8,4,0.0,0.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,59991,2025,37.0,7.627778,0.947368,0.833333,15,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,71.611111,1289,18,1.555556,28,2.055556,37,0.388889,0.166667,0.444444,7,3,8,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


In [15]:
with open("xgb_goalkeeper_model.pkl", "rb") as f:
    model_goalkeeper = pickle.load(f)

with open("xgb_defense_model.pkl", "rb") as f:
    model_defense = pickle.load(f)

with open("xgb_midfield_model.pkl", "rb") as f:
    model_midfield = pickle.load(f)

with open("xgb_offense_model.pkl", "rb") as f:
    model_offense = pickle.load(f)

print("Models loaded.")

Models loaded.


In [16]:
POSITION_GROUPS = {
    "goalkeeper": [
        "position_Torwart"
    ],
    "defense": [
        "position_Linker Verteidiger",
        "position_Abwehr",
        "position_Rechter Verteidiger",
        "position_Innenverteidiger"
    ],
    "midfield": [
        "position_Defensives Mittelfeld",
        "position_Linkes Mittelfeld",
        "position_Mittelfeld",
        "position_Offensives Mittelfeld",
        "position_Zentrales Mittelfeld",
        "position_Rechtes Mittelfeld"
    ],
    "offense": [
        "position_Hängende Spitze",
        "position_Linksaußen",
        "position_Mittelstürmer",
        "position_Rechtsaußen",
        "position_Sturm"
    ]
}

MODELS = {
    "goalkeeper": model_goalkeeper,
    "defense": model_defense,
    "midfield": model_midfield,
    "offense": model_offense
}

In [17]:
def assign_group(row):
    for group_name, cols in POSITION_GROUPS.items():
        existing_cols = [c for c in cols if c in row.index]
        if len(existing_cols) == 0:
            continue
        if row[existing_cols].sum() > 0:
            return group_name
    return np.nan

predict_dataset["model_group"] = predict_dataset.apply(assign_group, axis=1)

print(predict_dataset["model_group"].value_counts(dropna=False))
display(predict_dataset[["player_id", "season", "model_group"]].head())

model_group
midfield      567
defense       562
offense       430
goalkeeper    149
NaN           140
Name: count, dtype: int64


,player_id,season,model_group
0,33056,2025,midfield
1,37522,2025,midfield
2,44857,2025,offense
3,59232,2025,offense
4,59991,2025,midfield


In [18]:
prediction_frames = []

for group_name, model in MODELS.items():
    subset = predict_dataset[predict_dataset["model_group"] == group_name].copy()

    if subset.empty:
        print(f"{group_name}: keine Spieler")
        continue

    model_features = list(model.feature_names_in_)

    X_group = subset.copy()

    # Fehlende Spalten ergänzen
    for col in model_features:
        if col not in X_group.columns:
            X_group[col] = 0

    # Nur Modellspalten und richtige Reihenfolge
    X_group = X_group[model_features].copy()

    preds = model.predict(X_group)

    pred_out = subset[["player_id", "season", "model_group"]].copy()
    pred_out["prediction"] = preds

    prediction_frames.append(pred_out)

    print(f"{group_name}: {len(pred_out)} Spieler predicted")

goalkeeper: 149 Spieler predicted
defense: 562 Spieler predicted
midfield: 567 Spieler predicted
offense: 430 Spieler predicted


In [19]:
if len(prediction_frames) > 0:
    predictions_df = pd.concat(prediction_frames, ignore_index=True)
else:
    predictions_df = pd.DataFrame(columns=["player_id", "season", "model_group", "prediction"])

print(predictions_df.shape)
display(predictions_df.head())

(1708, 4)


,player_id,season,model_group,prediction
0,64629,2025,goalkeeper,7.095396
1,85240,2025,goalkeeper,7.069327
2,111423,2025,goalkeeper,7.145782
3,155406,2025,goalkeeper,7.121180
4,159305,2025,goalkeeper,7.092789


In [20]:
players_output = players_raw.copy()

players_output = players_output.merge(
    predictions_df[["player_id", "prediction"]],
    on="player_id",
    how="left"
)

print(players_output.shape)
display(players_output.head())

(5602, 7)


,player_id,player_name,nationality,date_of_birth,height,position,prediction
0,883105,Aadil Alleheri,Togo,2005-10-24,NaN,Defensives Mittelfeld,6.755599
1,923834,Aaron Akalé,Frankreich,2005-04-20,1.84,Mittelstürmer,NaN
2,707663,Aaron Appiah,Schweiz,2003-05-18,1.83,Mittelstürmer,NaN
3,1060512,Aaron Moos,Schweiz,2006-02-21,1.87,Innenverteidiger,6.755682
4,1235193,Aaron Tchamda,Schweiz,2007-01-07,1.95,Defensives Mittelfeld,6.837719


In [21]:
players_output.to_csv("players_predictions.csv", index=False)
print("Saved as: players_predictions.csv")

Saved as: players_predictions.csv
